# 1

In [26]:
def cyclic_convolution(a, b):
    """
    cyclic convolution of two vectors a and b.
    equivalent to multiplying two polynomials in Z[x]/(x^n - 1).
    """
    n = len(a)
    result = [0] * n
    for i in range(n):
        for j in range(n):
            # The index is (i + j) mod n because x^n = 1
            result[(i + j) % n] += a[i] * b[j]
    return result

# Coefficients of g(x) = (x^4 + 3x^3 + x^2 + 2x + 3)(2x^4 + 5x^2 + 6x + 2)
# Vector representation: [constant, x^1, x^2, x^3, x^4]

# P1 = 3 + 2x + 1x^2 + 3x^3 + 1x^4
vec1 = [3, 2, 1, 3, 1]

# P2 = 2 + 6x + 5x^2 + 0x^3 + 2x^4
vec2 = [2, 6, 5, 0, 2]

print(f"Vector 1: {vec1}")
print(f"Vector 2: {vec2}")

# Perform the multiplication in Z[x]/(x^5 - 1)
result_vec = cyclic_convolution(vec1, vec2)

print(f"\nResult of (3, 2, 1, 3, 1) * (2, 6, 5, 0, 2) in Z[x]/(x^5 - 1):")
print(f"Result Vector: {result_vec}")

expected_vec = [31, 29, 35, 24, 31]
print(f"\nExpected Coefficients of g(x) mod (x^5 - 1): {expected_vec}")

if result_vec == expected_vec:
    print("\nThe resulting vector matches the coefficients of g(x)")

Vector 1: [3, 2, 1, 3, 1]
Vector 2: [2, 6, 5, 0, 2]

Result of (3, 2, 1, 3, 1) * (2, 6, 5, 0, 2) in Z[x]/(x^5 - 1):
Result Vector: [31, 29, 35, 24, 31]

Expected Coefficients of g(x) mod (x^5 - 1): [31, 29, 35, 24, 31]

The resulting vector matches the coefficients of g(x)


$$
\begin{aligned}
g(x) &= (x^4 + 3x^3 + x^2 + 2x + 3)(2x^4 + 5x^2 + 6x + 2) \pmod{x^5 - 1} \\
\text{Expanding by terms of the first polynomial:} \\
x^4(2x^4 + 5x^2 + 6x + 2) &= 2x^8 + 5x^6 + 6x^5 + 2x^4 \equiv 2x^3 + 5x + 6 + 2x^4 \pmod{x^5 - 1} \\
3x^3(2x^4 + 5x^2 + 6x + 2) &= 6x^7 + 15x^5 + 18x^4 + 6x^3 \equiv 6x^2 + 15 + 18x^4 + 6x^3 \pmod{x^5 - 1} \\
x^2(2x^4 + 5x^2 + 6x + 2) &= 2x^6 + 5x^4 + 6x^3 + 2x^2 \equiv 2x + 5x^4 + 6x^3 + 2x^2 \pmod{x^5 - 1} \\
2x(2x^4 + 5x^2 + 6x + 2) &= 4x^5 + 10x^3 + 12x^2 + 4x \equiv 4 + 10x^3 + 12x^2 + 4x \pmod{x^5 - 1} \\
3(2x^4 + 5x^2 + 6x + 2) &= 6x^4 + 15x^2 + 18x + 6 \equiv 6x^4 + 15x^2 + 18x + 6 \pmod{x^5 - 1}
\end{aligned}
$$

$$
\begin{aligned}
\text{Summing the coefficients for each power of } x: \\
\text{Constant term: } & 6 + 15 + 4 + 6 = 31 \\
\text{Coefficient of } x^1: & 5 + 2 + 4 + 18 = 29 \\
\text{Coefficient of } x^2: & 6 + 2 + 12 + 15 = 35 \\
\text{Coefficient of } x^3: & 2 + 6 + 6 + 10 = 24 \\
\text{Coefficient of } x^4: & 2 + 18 + 5 + 6 = 31
\end{aligned}
$$

$$ g(x) \equiv 31x^4 + 24x^3 + 35x^2 + 29x + 31 \pmod{x^5 - 1} $$


# 2
## implementation from scratch

In [27]:
def clean(p):
    p = list(p)
    while len(p) > 1 and p[-1] == 0:
        p.pop()
    return p

def poly_sub(a, b, q):
    mx = max(len(a), len(b))
    return clean([((a[i] if i < len(a) else 0) - (b[i] if i < len(b) else 0)) % q for i in range(mx)])

def poly_mul(a, b, q):
    res = [0] * (len(a) + len(b) - 1)
    for i, ca in enumerate(a):
        for j, cb in enumerate(b):
            res[i+j] = (res[i+j] + ca * cb) % q
    return clean(res)

def poly_divmod(a, b, q):
    a, b = clean(a), clean(b)
    if b == [0]: raise ZeroDivisionError
    if len(a) < len(b): return [0], a
    
    q_poly = [0] * (len(a) - len(b) + 1)
    inv_b = pow(b[-1], -1, q)
    
    while len(a) >= len(b) and a != [0]:
        diff = len(a) - len(b)
        coef = (a[-1] * inv_b) % q
        q_poly[diff] = coef
        
        sub_term = [0] * diff + [(c * coef) % q for c in b]
        a = poly_sub(a, sub_term, q)
        
    return clean(q_poly), clean(a)

def poly_xgcd(a, b, q):
    x0, x1 = [1], [0]
    y0, y1 = [0], [1]
    
    while b != [0]:
        quot, rem = poly_divmod(a, b, q)
        a, b = b, rem
        x0, x1 = x1, poly_sub(x0, poly_mul(quot, x1, q), q)
        y0, y1 = y1, poly_sub(y0, poly_mul(quot, y1, q), q)
        
    inv_lc = pow(a[-1], -1, q)
    a = [(c * inv_lc) % q for c in a]
    x0 = [(c * inv_lc) % q for c in x0]
    y0 = [(c * inv_lc) % q for c in y0]
    
    return a, x0, y0

def poly_to_string(poly):
    if poly == [0]: return "0"
    terms = []
    for i, coeff in enumerate(poly):
        if coeff == 0: continue
        if i == 0: terms.append(f"{coeff}")
        elif i == 1: terms.append(f"{coeff}x")
        else: terms.append(f"{coeff}x^{i}")
    return " + ".join(terms)

f = [1, 3, 0, 0, 0, 3] 
m = [10, 0, 0, 0, 0, 0, 0, 1]
q = 11

gcd, inv_mod, _ = poly_xgcd(f, m, q)

print("Inverse in standard mod 11:")
print(poly_to_string(inv_mod))

inv_mods = [c if c <= 5 else c - 11 for c in inv_mod]

print("\nInverse in symmetric mods 11:")
print(poly_to_string(inv_mods))

Inverse in standard mod 11:
3 + 1x + 6x^2 + 4x^3 + 8x^4 + 8x^6

Inverse in symmetric mods 11:
3 + 1x + -5x^2 + 4x^3 + -3x^4 + -3x^6


### implementation using sympy

In [28]:
from sympy import symbols, Poly

x = symbols('x')

# Add symmetric=False to both polynomials
f = Poly(3*x**5 + 3*x + 1, x, modulus=11, symmetric=False)
m = Poly(x**7 - 1, x, modulus=11, symmetric=False)

# Calculate inverse
inv_mod = f.invert(m)

print("Inverse in standard mod 11:")
print(poly_to_string(inv_mod.all_coeffs()[::-1]))

# Add symmetric=False to both polynomials
f = Poly(3*x**5 + 3*x + 1, x, modulus=11, symmetric=True)
m = Poly(x**7 - 1, x, modulus=11, symmetric=True)

# Calculate inverse
inv_mod = f.invert(m)

print("Inverse in symmetric mods 11:")
print(poly_to_string(inv_mod.all_coeffs()[::-1]))

Inverse in standard mod 11:
3 + 1x + 6x^2 + 4x^3 + 8x^4 + 8x^6
Inverse in symmetric mods 11:
3 + 1x + -5x^2 + 4x^3 + -3x^4 + -3x^6
